# Overview

This is notebook for ResNet model training.

## 1. Imports / Settings

In [ ]:
import os
import datetime
import json

import cv2
import timm
import torch

import numpy as np
import pandas as pd
import torch.nn as nn
import torch.optim as optim
import pyarrow.parquet as pq
import pytorch_lightning as pl
import matplotlib.pyplot as plt

from torch.utils.data import Dataset, DataLoader
from pytorch_lightning.callbacks import Callback, EarlyStopping

from pyprojroot import here
from torch.mps import is_available

In [ ]:
PROJECT_ROOT = here()

INPUT_PATH = "data/meta/interim/"
IMAGES_PATH = "data/images/"
TARGET_FILE = "metadata_augmented.parquet"

OUTPUT_PATH = "output/resnet/"
MODELS_FOLDER = "models/"
META_FOLDER = "meta/"
MODEL_TYPE = "ResNet"
MODEL_FILENAME = "[{0}] {1}.ckpt"
META_FILENAME = "[{0}] {1}.txt"

TARGET_COLUMNS = [
    "sandstone_sludge", "siltstone_sludge", "argillite_sludge",
    "radiolarite_sludge", "coal_sludge",
    "limestone_sludge", "clay_sludge",
    "other_sludge"
]
SLUDGE_IMAGE_PATH_COLUMN = "sludge_image_path"

## 2. Classes Declaration

In [3]:
class SludgeDataset(Dataset):
    def __init__(self, df: pd.DataFrame, img_dir: str, target_cols: list):
        self.df = df.reset_index(drop = True)
        self.img_dir = img_dir
        self.target_cols = target_cols
    
    def __len__(self) -> int:
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, row[SLUDGE_IMAGE_PATH_COLUMN])
        if not os.path.exists(img_path):
            print(f"!!! WARN: Image (f{img_path}) doesn't exist or not available. !!!")

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        targets = row[self.target_cols].values.astype(np.float32)

        image = np.transpose(image, (2, 0, 1)).astype(np.float32) / 255.0

        return torch.tensor(image), torch.tensor(targets)

In [ ]:
class SludgeResNetLightning(pl.LightningModule):
    def __init__(self, model_name: str, num_classes: int, lr: float):
        super().__init__()
        self.save_hyperparameters()

        self.backbone = timm.create_model(model_name, 
                                          pretrained = model_training_params["pretrained"], 
                                          num_classes = 0)
        self.head = nn.Sequential(
            nn.Linear(self.backbone.num_features, HEAD_HIDDEN_SIZE),
            nn.LayerNorm(HEAD_HIDDEN_SIZE),
            nn.ReLU(),
            nn.Dropout(DROPOUT),
            nn.Linear(HEAD_HIDDEN_SIZE, num_classes)
        )
        self.softmax = nn.Softmax(dim = 1)
        self.criterion = nn.HuberLoss(delta = HUBER_DELTA)
        self.mae_metric = nn.L1Loss()
    
    def forward(self, x):
        features = self.backbone(x)
        logits = self.head(features)

        return self.softmax(logits) * 100
    
    def training_step(self, batch, batch_idx):
        images, targets = batch
        preds = self(images)
        loss = self.criterion(preds, targets)

        self.log("train_loss", 
                 loss, 
                 on_step = False, 
                 on_epoch = True, 
                 prog_bar = True, 
                 logger = True)
        return loss
    
    def validation_step(self, batch, batch_idx):
        images, targets = batch
        preds = self(images)
        loss = self.criterion(preds, targets)
        mae = self.mae_metric(preds, targets)

        self.log("val_loss", loss, on_epoch = True, prog_bar = True, logger = True)
        self.log("val_mae", mae, on_epoch = True, prog_bar = True, logger = True)

        for i, col_name in enumerate(self.trainer.datamodule.target_cols):
            class_mae = torch.mean(torch.abs(preds[:, i] - targets[:, i]))
            self.log(f"val_mae_{col_name.split("_")[0]}", class_mae, on_epoch = True, logger = True)
        return loss

    def configure_optimizers(self):
        optimizer = optim.AdamW(self.parameters(), lr = self.hparams.lr, weight_decay = WEIGHT_DECAY)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode = "min", factor = LR_FACTOR, patience = LR_PATIENCE
        )

        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "monitor": "val_loss"
            }
        }


In [ ]:
class SludgeDataModule(pl.LightningDataModule):
    def __init__(self, df: pd.DataFrame, img_dir: str, target_cols: list, batch_size: int, num_workers: int, target_well_id_for_validation: int):
        super().__init__()
        self.df = df
        self.img_dir = img_dir
        self.target_cols = target_cols
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.target_well_id_for_validation = target_well_id_for_validation


    def setup(self, stage = None):
        unique_wells = self.df['well_id'].unique()
        val_wells = [unique_wells[self.target_well_id_for_validation]]
        
        train_df = self.df[~self.df['well_id'].isin(val_wells)].reset_index(drop = True)
        val_df = self.df[
            self.df['well_id'].isin(val_wells) & (self.df['is_augmented'] == False)
        ].reset_index(drop = True)
        
        print(f"Well IDs (train): {train_df['well_id'].unique()}.")
        print(f"Well IDs (validation): {val_df['well_id'].unique()}.")
        
        self.train_dataset = SludgeDataset(train_df, self.img_dir, self.target_cols)
        self.val_dataset = SludgeDataset(val_df, self.img_dir, self.target_cols)

    def train_dataloader(self):
        return DataLoader(self.train_dataset, 
                          batch_size = self.batch_size, 
                          shuffle = True, 
                          num_workers = self.num_workers, 
                          drop_last = True)

    def val_dataloader(self):
        return DataLoader(self.val_dataset, 
                          batch_size = self.batch_size, 
                          shuffle = False, 
                          num_workers = self.num_workers)

## 3. Training Pipeline

### 3.1. Target Variables / Constants Declaration

In [ ]:
model_training_params = {
    "model_name": "resnet50d",
    "pretrained": True,
    "head_hidden_size": 512,
    "dropout": 0.3,
    "loss": "HuberLoss",
    "huber_delta": 1.0,
    "optimizer": "AdamW",
    "batch_size": 1,
    "num_workers": 0,

    "max_epochs": 30,
    "learning_rate": 3e-4,
    "weight_decay": 1e-4,
    "lr_factor": 0.5,
    "lr_patience": 3,

    "monitor_target": "val_loss",
    "early_stop_patience": 6,

    "target_well_for_validation_id": 1
}

MODEL_NAME = model_training_params["model_name"]
HEAD_HIDDEN_SIZE = model_training_params["head_hidden_size"]
DROPOUT = model_training_params["dropout"]
HUBER_DELTA = model_training_params["huber_delta"]
BATCH_SIZE = model_training_params["batch_size"]

MAX_EPOCHS = model_training_params["max_epochs"]
LEARNING_RATE = model_training_params["learning_rate"]
WEIGHT_DECAY = model_training_params["weight_decay"]
LR_FACTOR = model_training_params["lr_factor"]
LR_PATIENCE = model_training_params["lr_patience"]

MONITOR_TARGET = model_training_params["monitor_target"]
EARLY_STOP_PATIENCE = model_training_params["early_stop_patience"]

NUM_WORKERS = model_training_params["num_workers"]

TARGET_WELL_ID_FOR_VALIDATION = model_training_params["target_well_for_validation_id"]

In [7]:
final_input_path_images = os.path.join(PROJECT_ROOT, IMAGES_PATH)
final_input_path_metadata = os.path.join(PROJECT_ROOT, INPUT_PATH, TARGET_FILE)

final_output_path = os.path.join(PROJECT_ROOT, OUTPUT_PATH)
final_output_path_models = os.path.join(final_output_path, MODELS_FOLDER)
final_output_path_meta = os.path.join(final_output_path, META_FOLDER)

os.makedirs(final_output_path_models, exist_ok = True)
os.makedirs(final_output_path_meta, exist_ok = True)

### 3.2. Training Process

In [8]:
def get_compute_engine():
    # CUDA is not available on macOS.
    # if torch.backends.cuda.is_available():
    #     compute_engine = "cuda"
    if torch.backends.mps.is_available():
        compute_engine = "mps"
    else:
        compute_engine = "cpu"
    
    return compute_engine

class MetricHistory(Callback):
    def __init__(self):
        super().__init__()
        self.epoch_metrics = []
        self.enabled = True

    def on_validation_epoch_end(self, trainer, pl_module):
        if trainer.sanity_checking or not self.enabled:
            return

        metrics = {
            name: float(value.detach().cpu()) if isinstance(value, torch.Tensor) else value
            for name, value in trainer.callback_metrics.items()
        }
        self.epoch_metrics.append({"epoch": trainer.current_epoch + 1, "metrics": metrics})

def save_training_metadata(metadata, filepath):
    with open(filepath, "w", encoding = "utf-8") as metadata_file:
        json.dump(metadata, metadata_file, indent = 2, ensure_ascii = False)
        metadata_file.write("\n")

def run_training():
    training_started_at = datetime.datetime.now().strftime("%Y-%m-%dT%H:%M:%S")
    print("Loading the target metadata...")
    metadata_df = pd.read_parquet(final_input_path_metadata)

    data_module = SludgeDataModule(
        df = metadata_df,
        img_dir = final_input_path_images,
        target_cols = TARGET_COLUMNS,
        batch_size = BATCH_SIZE,
        num_workers = NUM_WORKERS,
        target_well_id_for_validation = TARGET_WELL_ID_FOR_VALIDATION
    )
    model = SludgeResNetLightning(
        model_name = MODEL_NAME,
        num_classes = len(TARGET_COLUMNS),
        lr = LEARNING_RATE
    )
    metric_history = MetricHistory()
    callbacks = [
        metric_history,
        EarlyStopping(monitor = MONITOR_TARGET,
                    patience = EARLY_STOP_PATIENCE,
                    mode = "min",
                    verbose = True),
    ]

    compute_engine = get_compute_engine()
    trainer = pl.Trainer(
        max_epochs = MAX_EPOCHS,
        accelerator = compute_engine,
        devices = 1,
        callbacks = callbacks,
        logger = False,
        enable_checkpointing = False,
        log_every_n_steps = 5
    )

    trainer.fit(model, datamodule = data_module)

    # Calculate final metrics first, then save exactly one final model artifact.
    metric_history.enabled = False
    final_metrics = trainer.validate(model, datamodule = data_module, verbose = False)[0]
    training_completed_at = datetime.datetime.now().strftime("%Y-%m-%dT%H:%M:%S")

    model_filename = MODEL_FILENAME.format(training_completed_at, MODEL_TYPE)
    meta_filename = META_FILENAME.format(training_completed_at, MODEL_TYPE)
    model_filepath = os.path.join(final_output_path_models, model_filename)
    meta_filepath = os.path.join(final_output_path_meta, meta_filename)

    trainer.save_checkpoint(model_filepath)
    metadata = {
        "model_type": MODEL_TYPE,
        "model_file": model_filename,
        "training_started_at": training_started_at,
        "training_completed_at": training_completed_at,
        "hyperparameters": model_training_params,
        "fold_training_metrics": [{
            "fold": 1,
            "validation_well_index": TARGET_WELL_ID_FOR_VALIDATION,
            "epoch_metrics": metric_history.epoch_metrics
        }],
        "cross_validation_metrics": final_metrics,
        "final_training_metrics": final_metrics
    }
    save_training_metadata(metadata, meta_filepath)

    print(f"Final model saved to: {model_filepath}.")
    print(f"Training metadata saved to: {meta_filepath}.")
    return model, metadata

In [9]:
print("Starting the training cycle.")

model, training_metadata = run_training()

print("Training cycle completed.")

Starting the training cycle.
Loading the target metadata...


GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Well IDs (train): [4 2 3].
Well IDs (validation): [1].


┏━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name       ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ backbone   │ ResNet     │ 23.5 M │ train │     0 │
│ 1 │ head       │ Sequential │  1.1 M │ train │     0 │
│ 2 │ softmax    │ Softmax    │      0 │ train │     0 │
│ 3 │ criterion  │ HuberLoss  │      0 │ train │     0 │
│ 4 │ mae_metric │ L1Loss     │      0 │ train │     0 │
└───┴────────────┴────────────┴────────┴───────┴───────┘

Trainable params: 24.6 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 24.6 M                                                                                               
Total estimated model params size (MB): 98.326                                                                     
Modules in train mode: 237                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

...

Metric val_loss improved. New best score: 11.263


Metric val_loss improved by 2.680 >= min_delta = 0.0. New best score: 8.583


Metric val_loss improved by 1.873 >= min_delta = 0.0. New best score: 6.710


Monitored metric val_loss did not improve in the last 6 records. Best score: 6.710. Signaling Trainer to stop.


Training cycle completed.
